In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

fact = spark.table("gold_fact_production")
b = fact.agg(F.min("production_month").alias("lo"), F.max("production_month").alias("hi")).collect()[0]

dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(b["lo"]), F.last_day(F.lit(b["hi"])), F.expr("interval 1 day"))).alias("date"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("month_start", F.trunc("date", "month"))
)
dim_date.write.mode("overwrite").format("delta").saveAsTable("gold_dim_date")

# Optional: operator dimension (one row per BAID)
dim_operator = (
    fact.filter(F.col("OperatorBAID").isNotNull())
    .groupBy("OperatorBAID")
    .agg(F.max("OperatorName").alias("operator_name"))
)
dim_operator.write.mode("overwrite").format("delta").saveAsTable("gold_dim_operator")

StatementMeta(, 75661958-f6c0-4d63-bfe8-6f0544852276, 3, Submitted, Running, Running, True)

In [ ]:
%%sql
SELECT date_format(production_month, 'yyyy-MM') AS ym, SUM(volume) AS gas
FROM gold_fact_production
WHERE ProductID = 'GAS'
GROUP BY 1
ORDER BY 1

StatementMeta(, , -1, Waiting, , Waiting, True)